# Me GPT Notebook

## Importing all needed libraries

In [71]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.documents import Document
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain.chat_models import init_chat_model

from langsmith import Client

from pathlib import Path

from dotenv import load_dotenv

from IPython.display import Markdown

from dotenv import load_dotenv

import os

In [73]:
load_dotenv()
api_key = os.getenv("GOOGLE_API_KEY")

## Embedding Markdown files

Below we define a list where each element will be a text document, containing the text from each markdown file:

In [74]:
documents = []

for file_path in Path('knowledge').glob('*.md'):
    text = file_path.read_text(encoding='utf-8')

    documents.append(
        Document(
            page_content=text,
            metadata={'source':str(file_path)}
        )
    )
# At this point
#   Document 1 = projects.md
#   Document 2 = experience.md
#   Document 3 = skills.md

We define the splitter that will split each document from the list, following the hierarchy specified in 'headers':

In [75]:
# "#" represents heading level 1.
# "##" represents heading level 2.
headers = [
    ('#','category'),
    ('##','section')
]

markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers,
    strip_headers=False
)

We instantiate the list that will contain all sections, with the granularity defined in the markdown_splitter:

In [76]:
sections = []

for document in documents:
    file_sections = markdown_splitter.split_text(document.page_content)

    for section in file_sections:
        section.metadata['source'] = document.metadata['source']

    sections.extend(file_sections)

With the sections ready, we now want to split them into chunks, where each chunk contains parts of the previous one and the next one (chunk_overlap). This is because we need be able to keep some context from one chunk to another, so that the overall document has all parts related to one another, and a general context picture is kept.

In [77]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150
)

chunks = text_splitter.split_documents(sections)

We visualize the chunks:

In [78]:
for index, chunk in enumerate(chunks[:5]):
    print(f'\n--- Chunk {index} ---')
    print(chunk.metadata)
    print(chunk.page_content)


--- Chunk 0 ---
{'category': 'Skills', 'section': 'Demonstrated Technical and hard Skills', 'source': 'knowledge/skills.md'}
# Skills  
## Demonstrated Technical and hard Skills
Marc has demonstrated Python, pandas, scikit-learn, XGBoost, RoBERTa, ChromaDB,
Gemini, FastAPI, Streamlit, natural language processing (NLP), classification,
retrieval-augmented generation (RAG), data preprocessing, and model evaluation.  
These skills are supported by the LIAR Political-Statement Classification Project and
the Marc GPT Online CV Project.  
Other skills Marc has demonstrated in his professional life: Looker, DBT (notably Y42 and Snowflake), GCP (notably Google BigQuery), data pipeline management and orchestration, Google Looker, Metabase.

--- Chunk 1 ---
{'category': 'Skills', 'section': 'Demonstrated Technical and hard Skills', 'source': 'knowledge/skills.md'}
Marc is also very familiar with Software as a aservice solar monitoring platforms, which he has developped in his masters internship

## Embedding

First, we define embedding function:

In [ ]:
embeddings = GoogleGenerativeAIEmbeddings(
    model='models/gemini-embedding-001',
    api_key=api_key
)

Then, create the Chroma vector store where we will embed our documents:

In [80]:
vector_store = Chroma(
    collection_name='ep_plenary',
    embedding_function=embeddings,
    persist_directory='./chroma_ep_follower'
)

Finally, we embed the documents to the vector store:

In [81]:
docs_embedded = vector_store.add_documents(documents=chunks)

## Querying to LLM

### Define query and retrieve similar sections

In [82]:
# We start by defining the query
query = "What are some of Marc's projects?"

# We continue by retrieving similar docs to the query from the document
retrieved_docs = vector_store.similarity_search(query, k=5)

# We group the retrieved docs to create the context later in the prompt
context = "\n\n".join(doc.page_content for doc in retrieved_docs)

### Defining prompt

In [83]:
from langchain_core.prompts import ChatPromptTemplate

# We create the prompt_template and then the prompt
client = Client()

system_msg = """
Answer from the provided context.
Do not invent information.
If the answer is unavailable, say so clearly.
"""

prompt_template = ChatPromptTemplate.from_messages([
    ('system', system_msg),
    ('human', f"""
Context:
{context}

Question:
{query}
"""),
])
# We define the prompt
prompt = prompt_template.invoke(
    {"context": context, "question": query}
).to_messages()

### Instantiating LLM and sending prompt

In [87]:
llm = init_chat_model(
    "gemini-2.5-flash-lite",
    model_provider="google_genai",
    api_key=api_key,
    max_tokens=1_000)

answer = llm.invoke(prompt)

### Displaying answer from LLM

In [88]:
Markdown(answer.content)

Marc's projects include:
* Seeding Impact (Co-Founder)
* LIAR Political-Statement Classification Project (Data Scientist)
* Marc GPT Online CV Project (Creator and developer)